# RF implementation
## Author: Hayden Mann

### RF
1. Install and Imports
2. Import matchups
3. Preliminary run on all features
4. Feature importance of each feature in partitioning
5. Correlation Matrix
6. Mathematically trim features. New Correlation Matrix
7. Get trimmed feature correlation with chlorophyll-a
8. Train on trimmed features!!!

- RF installment is based off of Corcoran, F., & Parrish, C. E. (2021). Diffuse Attenuation Coefficient (Kd) from ICESat-2 ATLAS Spaceborne Lidar Using Random-Forest Regression

### Plots
1. Chl v Chl scatter plot
2. Variable scatter plots
3. Small-region time series

In [ ]:
import os
import sys
import glob
import shutil
import warnings
from pathlib import Path
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings("ignore")


import numpy as np
import pandas as pd


import geopandas as gpd
import rasterio
import rioxarray as rxr
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import LineString, Polygon
from pyproj import Transformer
from scipy.spatial import cKDTree
from scipy.stats import skew, kurtosis, pearsonr


import xarray as xr
import earthaccess as ea
from sliderule import icesat2, sliderule
from harmony import (
    BBox,
    CapabilitiesRequest,
    Client,
    Collection,
    JobsRequest,
    LinkType,
    Request,
)


import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
import contextily as cx
import seaborn as sns
from matplotlib.colors import LogNorm, TwoSlopeNorm
from matplotlib.ticker import FuncFormatter, StrMethodFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


from aux_fx_process import (
    grid_data,
    add_lon_lat,
    get_photons,
    water_mask,
    water_surface_retrieval_and_binning,
    get_segment_vars,
    get_PACE_time_windows,
    get_PACE_data,
    gridded_chl,
    get_segment_centers,
    match_chl_to_segments,
)
from aux_fx_plot import ATLAS_overlays_TX, make_scatter_plot


ea.login()
# ea.login(strategy="interactive", persist=True)

In [ ]:
""" 

Import Matchups 

"""
import glob
import geopandas as gpd
import pandas as pd

# Read the raw parquet collections
df_filtered_raw = pd.concat(
    [
        pd.read_parquet(f)
        for f in glob.glob("MATCHUPS/GULF_DATA_FILTERED/data._*")
    ],
    ignore_index=True,
)
df_raw = pd.concat(
    [pd.read_parquet(f) for f in glob.glob("MATCHUPS/GULF_DATA_INPACEGAPS/data._*")],
    ignore_index=True,
)

# Extract standard string dates
df_filtered_raw["date"] = pd.to_datetime(df_filtered_raw["time"]).dt.date
df_raw["date"] = pd.to_datetime(df_raw["time"]).dt.date

# Decode the WKB binary column back into actual LineStrings
df_filtered_raw["geo"] = gpd.GeoSeries.from_wkb(df_filtered_raw["geo"])
df_raw["geo"] = gpd.GeoSeries.from_wkb(df_raw["geo"])

# Convert to proper GeoDataFrames
matchup_df_filtered = gpd.GeoDataFrame(
    df_filtered_raw, geometry="geo", crs="EPSG:4326"
)
matchup_df = gpd.GeoDataFrame(df_raw, geometry="geo", crs="EPSG:4326")

print("Filtered Matchup Date Counts:")
print(matchup_df_filtered["date"].value_counts().head())

In [ ]:
""" Preliminary RF run on all features """

# calculate the features needed
matchup_df_filtered["log_R_sw"] = np.log10(matchup_df_filtered["R_sw"])
matchup_df_filtered["D75/D25"] = matchup_df_filtered["D75"] / matchup_df_filtered["D25"]
matchup_df_filtered["log_chl_a"] = np.log10(matchup_df_filtered["chl_a"])


features = [
    'D10', 'D25', 'D50', 'D75', 'D90', 'N_surface',
    'N_subsurface', 'log_R_sw',
    'z_max', 'mean_depth', 'std_of_depth',
    'skewness', 'kurtosis', 'D75/D25'
]

# Set our X: LiDAR, and Y: PACE
X = matchup_df_filtered[features]
y = matchup_df_filtered["log_chl_a"]

# Split data. random_state 42 was selected by a random number generator
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# train RF:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42
)

rf.fit(X_train, y_train)

""". 

Get score

"""

from sklearn.metrics import r2_score

y_pred = rf.predict(X_test)

print("R²:", r2_score(y_test, y_pred))

# SAVE: toggle as needed
# joblib.dump(rf, 'RF_ALLFEATS.joblib')

# LOAD: toggle as needde
# rf = joblib.load('RF_JULY14_ALLFEATS.joblib')

In [ ]:
""" Feature importance from the random forest"""

# Calculate feature importance 
feature_importance = pd.Series(
    rf.feature_importances_,
    index=features
).sort_values(ascending=False)

print(feature_importance)

# Plot
feature_importance.sort_values().plot(
    kind="barh",
    figsize=(8,5)
)

plt.xlabel("Relative Feature Importance")
plt.title("Random Forest Feature Importance")
plt.show()

In [ ]:
""" Correlation Matrix of ALL FEATURES """

import seaborn as sns
import matplotlib.pyplot as plt

# Calculate correlation matrix
corr_matrix = matchup_df_filtered[features].corr()

# Plot heatmap
plt.figure(figsize=(12,10))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
""" 
Trim features from the dataset. Comparing pairs of 
features with a correlation greater than abs(+/-0.75). Remove the feature in the 
pair with a lower feature importance.


Idea from Corcoran, F., & Parrish, C. E. (2021). 
Diffuse Attenuation Coefficient (Kd) from ICESat-2 
ATLAS Spaceborne Lidar Using Random-Forest Regression
"""

# find highly correlated pairs
import numpy as np
corr_matrix = matchup_df_filtered[features].corr()

# Find pairs with |correlation| > 0.75
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_value = corr_matrix.iloc[i, j]
        
        if abs(corr_value) > 0.75:
            high_corr_pairs.append(
                (
                    corr_matrix.columns[i],
                    corr_matrix.columns[j],
                    corr_value
                )
            )

print("Highly correlated pairs:")
for pair in high_corr_pairs:
    print(pair)
    features_trimmed = features.copy()

# remove the less important of the correlated features
for feat1, feat2, corr in high_corr_pairs:
    
    # Compare importance
    if feature_importance[feat1] < feature_importance[feat2]:
        remove = feat1
    else:
        remove = feat2
    
    if remove in features_trimmed:
        features_trimmed.remove(remove)
        print(f"Removed {remove} (correlation = {corr:.2f})")

print("\nRemaining features:")
print(features_trimmed)

# Calculate correlation matrix
corr_matrix = matchup_df_filtered[features_trimmed].corr()

# Plot heatmap
plt.figure(figsize=(12,10))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("Feature Correlation Matrix: TRIMMED")
plt.tight_layout()
plt.show()


In [ ]:
""" Correlation with Chlorophyll """

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Compute correlation of all features with chlorophyll
chlorophyll_corr = (
    pd.concat(
        [matchup_df_filtered[features_trimmed],
         matchup_df_filtered["log_chl_a"]],
        axis=1
    )
    .corr()["log_chl_a"]          
    .sort_values(ascending=False)
    .drop("log_chl_a")
)

print(chlorophyll_corr)
plt.figure(figsize=(6, 8))

sns.heatmap(
    chlorophyll_corr.to_frame(),
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f",
    cbar=True
)

plt.title("Correlation of Features with Chlorophyll")
plt.tight_layout()
plt.show()

In [ ]:
""" TRAIN ON TRIMMED FEATURES: THIS IS WHAT IS USED """
## retrain on optimal!
X_trimmed = matchup_df_filtered[features_trimmed]

X_train, X_test, y_train, y_test = train_test_split(
    X_trimmed,
    y,
    test_size=0.2,
    random_state=42
)

rf_trimmed = RandomForestRegressor(
    n_estimators=500,
    random_state=42
)

rf_trimmed.fit(X_train, y_train)

y_pred = rf_trimmed.predict(X_test)

print("R² after trimming:", r2_score(y_test, y_pred))

# Save: toggle as desired
# # joblib.dump(rf_trimmed, 'RF_JULY14_TRIMMEDFEATS.joblib')

# Load: toggle as desired
# rf_trimmed = joblib.load('RF_JULY14_TRIMMEDFEATS.joblib')

# Begin Plots

In [ ]:
"""
Plot Chl vs Chl. outliers toggled below
"""

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


""" IF YOU WANT OUTLIERS REMOVED OR NOT"""
outliers_removed = False
""" IF YOU WANT OUTLIERS REMOVED OR NOT"""


# Metrics (calculated in log10 space)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)


from scipy.stats import pearsonr

# Pearson correlation (calculated in log10 space)
corr, p_value = pearsonr(y_test, y_pred)
print(corr)
print(p_value)

# Plot
fig, ax = plt.subplots(figsize=(6, 6))
bias = np.mean(y_pred - y_test)

# Limits in log10 space
lims = [
    min(y_test.min(), y_pred.min()),
    max(y_test.max(), y_pred.max())
]

# 1:1 line
ax.plot(
    lims, lims,
    '--',
    color='black',
    linewidth=2,
    label="1:1"
)


if outliers_removed == True:
    # Remove extreme plot outliers only
    plot_residuals = y_pred - y_test

    threshold = np.percentile(np.abs(plot_residuals), 95)
    plot_mask = np.abs(plot_residuals) < threshold # adjust threshold as needed

    y_test_plot = y_test[plot_mask]
    y_pred_plot = y_pred[plot_mask]

    print(f"Removed {np.sum(~plot_mask)} points from plot only")


ax.scatter(
    y_test,
    y_pred,
    s=15,
    alpha=0.05,
    edgecolor="none",
    color="royalblue"
)

# Fit line / Regression line
if outliers_removed == True:
    coef = np.polyfit(y_test_plot, y_pred_plot, 1)
    xfit = np.linspace(y_test_plot.min(), y_test_plot.max(), 100)
else:
    coef = np.polyfit(y_test, y_pred, 1)
    xfit = np.linspace(*lims, 100)

ax.plot(
    xfit,
    coef[0]*xfit + coef[1],
    color="crimson",
    linewidth=2,
    label=f"Fit: y={coef[0]:.2f}x+{coef[1]:.2f}"
)

# Labels
ax.set_xlabel("PACE l2m Chl-a [mg m$^{-3}$]", fontsize=12)
ax.set_ylabel("ATL03 model-derived predicted Chl-a [mg m$^{-3}$]", fontsize=12)

# Set axis ticks in log space, but label them as real values
tick_values = np.array([0.01, 0.1, 1, 10, 100])
tick_positions = np.log10(tick_values)
ax.set_xticks(tick_positions)
ax.set_yticks(tick_positions)
ax.spines[['top', 'right']].set_visible(False)
ax.set_xticklabels(["0.01", "0.1", "1.0", "10", "100"])
ax.set_yticklabels(["0.01", "0.1", "1.0", "10", "100"])


# Metrics box
ax.text(
    0.05, 0.95,
    f"$R^2$ = {r2:.2f}\n"
    f"RMSE = {rmse:.2f} log$_{{10}}$(mg m$^{{-3}}$)\n"
    f"MAE = {mae:.2f} log$_{{10}}$(mg m$^{{-3}}$)\n"
    f"Bias = {bias:.5f} log$_{{10}}$(mg m$^{{-3}}$)\n"
    f"N = {len(y_test)}",
    transform=ax.transAxes,
    va="top",
    fontsize=11,
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="gray")
)

# Format
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.legend()

# Save figure
if outliers_removed == True:
    plt.tight_layout()
    plt.savefig(
        "FINAL_TX_FIGS/1kmTX_NN_OUTLIERSREMOVED.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.show()
else:
    plt.tight_layout()
    plt.savefig(
        "FINAL_TX_FIGS/1kmTX_NN.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.show()


In [ ]:
"""" MAP CHL under LIDAR Chl """
""" can use same/diff colormap, and toggle an outline around the transects"""

# date to plot
ten_most_dates = ["2025-08-12"] #["2025-08-12"]

from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.cm as cm
import matplotlib.colors as colors
import glob
import xarray as xr
from aux_fx_process import grid_data


# Default option
colormap = "viridis"
outline = False

""" OPTION: SAME Colormap"""
# colormap_option = "SameCM"
# if colormap_option == "SameCM":
#     colormap = "viridis"
#     outline = False

""" OPTION: OUTLINE: SAME Colormap"""
colormap_option = "OutlineSameCM"
if colormap_option == "OutlineSameCM":
    colormap = "viridis"
    outline = True

""" OPTION: DIFFERENT Colormap"""
# colormap_option = "DiffCM"
# if colormap_option == "DiffCM":
#     colormap = "margma"
#     outline = False


matchup_df["log_R_sw"] = np.log10(matchup_df["R_sw"])
matchup_df["D75/D25"] = matchup_df["D75"] / matchup_df["D25"]
matchup_df["log_chl_a"] = np.log10(matchup_df["chl_a"])

# Predict log10(chl-a) for EVERY LIDAR POINT. not just for matchups since the point is to fill gaps :)
matchup_df["pred_log_chl_a"] = rf_trimmed.predict(
    matchup_df[features_trimmed]
)

matchup_df["pred_chl_a"] = 10 ** matchup_df["pred_log_chl_a"]


# plot resolution
resolution = (0.015, 0.015)

# plot bounds
LAT_MIN = 25.911
LAT_MAX = 30.192
LON_MIN = -97.546
LON_MAX = -93.067
pLAT_MIN = LAT_MIN 
pLAT_MAX = LAT_MAX
pLON_MIN = LON_MIN
pLON_MAX = LON_MAX
xlim = ([pLON_MIN + resolution[0], pLON_MAX - resolution[0]])
ylim = ([pLAT_MIN + resolution[1], pLAT_MAX - resolution[1]])


# get PACE file per date
dict_date_files = {}

for date_str in ten_most_dates:
    # Convert to timestamp to do safe calendar math
    current_date = pd.to_datetime(date_str)
    pre_date_dt = current_date - pd.Timedelta(days=1)
    post_date_dt = current_date + pd.Timedelta(days=1)
    
    # Format them back to strings matching the PACE file naming system (YYYYMMDD)
    date_pace = current_date.strftime("%Y%m%d")
    pre_pace  = pre_date_dt.strftime("%Y%m%d")
    post_pace = post_date_dt.strftime("%Y%m%d")

    patterns = [
        f"DATA/PACE_l2_Texas/*{date_pace}*.nc",
        f"DATA/PACE_l2_Texas/*{pre_pace}T1[2-9]*.nc",
        f"DATA/PACE_l2_Texas/*{pre_pace}T2[0-4]*.nc",
        f"DATA/PACE_l2_Texas/*{post_pace}T0[0-9]*.nc",
        f"DATA/PACE_l2_Texas/*{post_pace}T1[0-2]*.nc"
    ]
    
    matched_files = []
    for pattern in patterns:
        matched_files.extend(glob.glob(pattern))
        
    dict_date_files[date_str] = matched_files


# instantiate plot
fig, ax = plt.subplots(
    figsize=(10, 8),
    constrained_layout=True
)

# plot parameters
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 8,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

# loop through all marked PACE files
i = 0
for date_key in dict_date_files:
    axis = ax

    gridded_list = []

    # convert into a gridded list of datasets
    for file in dict_date_files[date_key]:
        dt = xr.open_datatree(file)
        ds = xr.merge(dt.to_dict().values())
        ds = ds.set_coords(("longitude", "latitude"))

        ds_gridded = grid_data(ds, resolution)

        # Drop the original swath dimensions so xarray won't try to align them
        dims_to_drop = [d for d in ["number_of_lines", "pixels_per_line"] if d in ds_gridded.dims]
        coords_to_drop = [c for c in ["number_of_lines", "pixels_per_line"] if c in ds_gridded.coords]
        
        ds_gridded = ds_gridded.drop_dims(dims_to_drop).drop_vars(coords_to_drop, errors="ignore")

        gridded_list.append(ds_gridded)

    # plot
    combined = xr.concat(gridded_list, dim="swath")
    ds_gridded = combined.mean(dim="swath", skipna=True)

    chl_norm = LogNorm(vmin=0.01, vmax=10.0)
    im = ds_gridded["chlor_a"].plot(
        ax=axis,
        cmap="viridis",
        norm=chl_norm,
        add_colorbar=False,
    )

    # LIDAR overlay
    df_day_segments = matchup_df[
        pd.to_datetime(matchup_df['lidar_date']).dt.strftime("%Y-%m-%d") == date_key
    ]

    # If LiDAR data, then plot
    if len(df_day_segments) > 0:

        if outline == True:
            # LIDAR track outline
            df_day_segments.plot(
                ax=axis,
                color="white",
                linewidth=12,
                zorder=7,
            )

        df_day_segments.plot(
            ax=axis,
            column="pred_chl_a",
            cmap=colormap,
            linewidth=5,
            norm=chl_norm,
            legend=False,
            zorder= 8
        )
     

    # Land mask
    land = gpd.read_file("ne_10m_land.shp")
    land = land.to_crs(epsg=4326)
    land.plot(ax=axis, facecolor="lightgray", edgecolor="black", linewidth=0.5, zorder=50)

    # ATLAS Colorbar
    cax1 = inset_axes(
        axis, width="3%", height="100%", 
        loc="center left",
        bbox_to_anchor=(1.05, 0.0, 1, 1), 
        bbox_transform=axis.transAxes,
        borderpad=0,
    )
    cb1 = fig.colorbar(cm.ScalarMappable(norm=chl_norm, cmap=colormap), cax=cax1)
    cb1.set_label("ATLAS-derived Chl-a [mg m$^{-3}$]")

    # PACE Colorbar
    cax2 = inset_axes(
        axis, width="3%", height="100%",
        loc="center left",
        bbox_to_anchor=(1.22, 0.0, 1, 1),  # further right, next to cax1
        bbox_transform=axis.transAxes,
        borderpad=0,
    )
    cb2 = fig.colorbar(cm.ScalarMappable(norm=chl_norm, cmap="viridis"), cax=cax2)
    cb2.set_label("Chl-a [mg m$^{-3}$]")

    # Formatting
    axis.set_xlim([xlim[0] + resolution[0], xlim[1] - resolution[0]])
    axis.set_ylim([ylim[0] + resolution[1], ylim[1] - resolution[1]])
    axis.set_title("")
    axis.set_xlabel("Longitude", fontsize=16)
    axis.set_ylabel("Latitude", fontsize=16)

    # loop increment
    i = i+1


# Plot and Save

fig.canvas.draw()

bbox = fig.get_tightbbox(fig.canvas.get_renderer())

cb1.set_label("ATLAS-Predicted Chl-a [mg m$^{-3}$]")
cb2.set_label("PACE Chl-a [mg m$^{-3}$]")
fig.savefig(
    "FINAL_TX_FIGS/ChlvCHL_" + colormap_option + "" + ten_most_dates[0] + ".png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    pad_inches=0.2
)


plt.show()

# Scatter Plots

In [ ]:
from aux_fx_plot import make_scatter_plot

# Set parameters for scatter plot function
plot_configs = [
    dict(
        x_col="log_R_sw",
        xlabel='R$_{sw}$',
        xlim=[-2, 3],
        ylim=[-2, 3],
        savepath="FINAL_TX_FIGS/PACEChl_v_Rsw.png",
        text_xy=(0.62, 0.95),
        x_is_log=True,
    ),
    dict(
        x_col="skewness",
        xlabel="Skewness",
        xlim=[-20, 1],
        ylim=[-2, 2.5],
        savepath="FINAL_TX_FIGS/PACEChl_v_Skew.png",
    ),
    dict(
        x_col="kurtosis",
        xlabel="Kurtosis",
        xlim=[np.min(matchup_df_filtered["kurtosis"]), 100],
        ylim=[-2, 2.5],
        savepath="FINAL_TX_FIGS/PACEChl_v_Kurt.png",
    ),
    dict(
        x_col="N_subsurface",
        xlabel=r'N$_{subsurface}$',
        xlim=[20, 5000],
        ylim=[-2, 2.5],
        savepath="FINAL_TX_FIGS/PACEChl_v_NSurface.png",
    ),
]

# Plots
for cfg in plot_configs:
    make_scatter_plot(matchup_df_filtered, **cfg)
    print(matchup_df_filtered[cfg["x_col"]].describe())

# Fixed TS

In [ ]:
import os
import glob
import joblib
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
from shapely.geometry import Polygon

""" Load Data and RF Model. Note, need to get X_test, Y_test. Wasn't able to update in time"""
df_raw = pd.concat(
    [pd.read_parquet(f) for f in glob.glob("MATCHUPS/GULF_DATA_INPACEGAPS//data._*")],
    ignore_index=True,
)
df_raw["geo"] = gpd.GeoSeries.from_wkb(df_raw["geo"])
matchup_df = gpd.GeoDataFrame(df_raw, geometry="geo", crs="EPSG:4326")

rf_trimmed = joblib.load("RF_JULY14_TRIMMEDFEATS.joblib")

features_trimmed = ['D10', 'D50', 'N_surface', 'N_subsurface', 'log_R_sw',
                     'z_max', 'std_of_depth', 'D75/D25']

# Full bounds
LAT_MIN = 25.911
LAT_MAX = 30.192
LON_MIN = -97.546
LON_MAX = -93.067


# Small regional bounds
aoi = [[-94.9334460784,28.184748032],[-95.4886387605,28.8139127035],[-94.7901151948,29.2874588819],[-94.2349225127,28.6611476263],[-94.9334460784,28.184748032]]

aoi_poly = Polygon(aoi)

# Limit matchups to the small regional bounds
matchup_df = matchup_df[matchup_df.geometry.intersects(aoi_poly)]

# calculate necessary variables
matchup_df["log_R_sw"] = np.log10(matchup_df["R_sw"])
matchup_df["D75/D25"] = matchup_df["D75"] / matchup_df["D25"]

# Mini processing to match 01_match_maker
THRESHOLD_LIMIT = 200
type_a = matchup_df.loc[matchup_df["N_subsurface"] < THRESHOLD_LIMIT]
matchup_df = matchup_df.loc[matchup_df["N_subsurface"] >= THRESHOLD_LIMIT]

# Predict chl_a from LiDAR based on model (already trained)
matchup_df["pred_log_chl_a"] = rf_trimmed.predict(matchup_df[features_trimmed])
matchup_df["pred_chl_a"] = 10 ** matchup_df["pred_log_chl_a"]

# Sort matchups by time
matchup_df = matchup_df.sort_values(by="time")
matchup_df["lidar_date"] = pd.to_datetime(matchup_df["time"])

# Hourly aggregation of ATLAS-predicted chl-a
matchup_df["hour"] = matchup_df["lidar_date"].dt.floor("h")

# compute stats per hour
hourly_lidar_stats = (
    matchup_df.groupby("hour")["pred_chl_a"]
    .agg(["mean", "min", "max", "count"])
)

hourly_lidar_stats["pos_error"] = hourly_lidar_stats["max"] - hourly_lidar_stats["mean"]
hourly_lidar_stats["neg_error"] = hourly_lidar_stats["mean"] - hourly_lidar_stats["min"]

MIN_SEGMENTS = 3
hourly_lidar_stats = hourly_lidar_stats[hourly_lidar_stats["count"] >= MIN_SEGMENTS]

In [ ]:

""" 

Bounding Box plot: sanity check

"""
fig, ax = plt.subplots(figsize=(8, 8))

bbox_x = [LON_MIN, LON_MAX, LON_MAX, LON_MIN, LON_MIN]
bbox_y = [LAT_MIN, LAT_MIN, LAT_MAX, LAT_MAX, LAT_MIN]
ax.plot(bbox_x, bbox_y, color="red", linewidth=2, label="Bounding Box")

hx, hy = aoi_poly.exterior.xy
ax.plot(hx, hy, color="green", linewidth=2, label="AOI")

ax.set_xlim(LON_MIN, LON_MAX)
ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.set_aspect("equal")
plt.show()

In [ ]:
""" 

GET PACE DATA FOR THE YEAR. Downloaded not streamed. ETA ~8 minutes

"""
import earthaccess as ea
import xarray as xr


ea.login()
# ea.login(strategy='interactive', persist=True)

START_DATE = "2025-01-01"
END_DATE = "2025-12-31"

results = ea.search_data(
    short_name="PACE_OCI_L2_BGC",
    bounding_box=(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
    temporal=(START_DATE, END_DATE),
    count=False
)
dfs = []

import earthaccess as ea
import xarray as xr
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
from concurrent.futures import ThreadPoolExecutor, as_completed

ea.login()


results = ea.search_data(
    short_name="PACE_OCI_L2_BGC",
    bounding_box=(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
    temporal=(START_DATE, END_DATE),
    count=False
)

# Download locally instead of streaming — much fewer, larger reads
files = ea.download(results, local_path="./pace_downloads/GULF")

def process_file(args):
    fpath, result = args
    try:
        with xr.open_dataset(fpath, group="geophysical_data") as data, \
             xr.open_dataset(fpath, group="navigation_data") as nav:

            lat = nav["latitude"].values
            lon = nav["longitude"].values
            chl = data["chlor_a"].values  # pull to numpy immediately

        mask = (
            (lat >= LAT_MIN) & (lat <= LAT_MAX) &
            (lon >= LON_MIN) & (lon <= LON_MAX) &
            ~pd.isna(chl)
        )
        if not mask.any():
            return None

        df = pd.DataFrame({
            "chlor_a": chl[mask],
            "latitude": lat[mask],
            "longitude": lon[mask],
        })
        df["swath_time"] = pd.to_datetime(
            result["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
        )
        return df
    except Exception as e:
        print(f"Skipping {fpath}: {e}")
        return None

dfs = []
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = [ex.submit(process_file, (f, r)) for f, r in zip(files, results)]
    for fut in as_completed(futures):
        result = fut.result()
        if result is not None:
            dfs.append(result)

df = pd.concat(dfs, ignore_index=True)
chl_a_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs="EPSG:4326",
)
chl_a_gdf

chl_a_gdf.to_parquet("2025GULFDATA", index=False)

In [ ]:
""" 

Load in PACE data. Compute stats for area.

"""

chl_a_gdf = gpd.read_parquet("2025GULFDATA")

chl_a_gdf = chl_a_gdf.sort_values("swath_time")

aoi = [[-94.9334460784,28.184748032],[-95.4886387605,28.8139127035],[-94.7901151948,29.2874588819],[-94.2349225127,28.6611476263],[-94.9334460784,28.184748032]]


points = Polygon(aoi)

chl_a_gdf = chl_a_gdf[chl_a_gdf.geometry.intersects(points)]

chl_a_gdf["swath_hour"] = chl_a_gdf["swath_time"].dt.floor("h")

swath_area_stats = (
    chl_a_gdf.groupby("swath_hour")["chlor_a"]
    .agg(["mean", "min", "max", "count"])
)

swath_area_stats["pos_error"] = swath_area_stats["max"] - swath_area_stats["mean"]
swath_area_stats["neg_error"] = swath_area_stats["mean"] - swath_area_stats["min"]

swath_area_stats

In [ ]:
""" 

Plot time series

"""


import os
import glob
import joblib
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
from shapely.geometry import Polygon


plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 11

fig, ax = plt.subplots(1, 1, figsize=(9, 5), constrained_layout=True)

pace_lower = swath_area_stats["mean"] - swath_area_stats["neg_error"]
pace_upper = swath_area_stats["mean"] + swath_area_stats["pos_error"]

""" Smooth PACE with a centered rolling mean """
window = 1  # 1 is not smoothed at all

pace_mean = (
    swath_area_stats["mean"]
    .rolling(window=window, center=True, min_periods=1)
    .mean()
)

pace_lower = (
    (swath_area_stats["mean"] - swath_area_stats["neg_error"])
    .rolling(window=window, center=True, min_periods=1)
    .mean()
)

pace_upper = (
    (swath_area_stats["mean"] + swath_area_stats["pos_error"])
    .rolling(window=window, center=True, min_periods=1)
    .mean()
)

# shading for PACE errors
ax.fill_between(
    swath_area_stats.index,
    pace_lower,
    pace_upper,
    color="#2E86C1",
    alpha=0.85,
    linewidth=0,
    label="PACE uncertainty",
    zorder=1,
)

# PACE plots
ax.plot(
    swath_area_stats.index,
    pace_mean,
    "o-",
    markersize=7.5,
    markeredgecolor="white",
    markeredgewidth=1.1,
    linewidth=2.8,
    alpha=1.0,
    label="_nolegend_",
    color="#2E86C1",
    zorder=2,
)

# Differentiate between nighttime and daytime data
day_mask = (hourly_lidar_stats.index.hour >= 6) & (hourly_lidar_stats.index.hour < 20)
night_mask = ~day_mask

# LiDAR plot
ax.plot(
    hourly_lidar_stats.index,
    hourly_lidar_stats["mean"],
    "-",
    color="#F27411",
    linewidth=3.0,
    alpha=0.95,
    zorder=3,
    label="_nolegend_",
)

# LiDar plot for daytime
day_err = ax.errorbar(
    hourly_lidar_stats.index[day_mask],
    hourly_lidar_stats["mean"][day_mask],
    yerr=[
        hourly_lidar_stats["neg_error"][day_mask],
        hourly_lidar_stats["pos_error"][day_mask],
    ],
    fmt="o",
    markersize=10,
    markeredgecolor="white",
    markeredgewidth=1.3,
    capsize=5,
    capthick=2.7,
    elinewidth=2.4,
    alpha=0.95,
    color="#F27411",
    ecolor="#F27411",
    label="ATLAS Model-Derived Chl-a (day)",
    zorder=4,
)

# update path effects
for line in day_err.lines[1]:
    line.set_path_effects([pe.Stroke(linewidth=3.4, foreground="white"), pe.Normal()])
for cap in day_err[2]:
    cap.set_path_effects([pe.Stroke(linewidth=3.4, foreground="white"), pe.Normal()])

# Lidar plot for daytime
night_err = ax.errorbar(
    hourly_lidar_stats.index[night_mask],
    hourly_lidar_stats["mean"][night_mask],
    yerr=[
        hourly_lidar_stats["neg_error"][night_mask],
        hourly_lidar_stats["pos_error"][night_mask],
    ],
    fmt="^",
    markersize=11,
    markeredgecolor="white",
    markeredgewidth=1.3,
    capsize=5,
    capthick=2.7,
    elinewidth=2.4,
    alpha=0.95,
    color="#F27411",
    ecolor="#F27411",
    label="ATLAS Model-Derived Chl-a (night)",
    zorder=4,
)
# update path effects
for line in night_err.lines[1]:
    line.set_path_effects([pe.Stroke(linewidth=3.4, foreground="white"), pe.Normal()])
for cap in night_err[2]:
    cap.set_path_effects([pe.Stroke(linewidth=3.4, foreground="white"), pe.Normal()])

# format
ax.set_yscale("log")
ax.set_ylim([0.1, 100])
ax.set_ylabel("Coastal-Averaged Chl-a [mg m$^{-3}$]", fontsize=12)
ax.set_xlabel("Date", fontsize=12)

# plot limits
pSTART_DATE = "2025-01-01"
pEND_DATE = "2025-12-31"
ax.set_xlim(
    pd.Timestamp(pSTART_DATE),
    pd.Timestamp(pEND_DATE),
)

# More formatting
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.6)
ax.spines["bottom"].set_linewidth(1.6)
ax.set_yticks([0.1, 1, 10, 100])
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.tick_params(axis="both", which="major", labelsize=10, length=5, width=1.5)


legend = ax.legend(
    frameon=True,
    framealpha=0.9,
    edgecolor="black",
    fontsize=9.5,
    loc="upper right",
)
legend.get_frame().set_facecolor("white")


# Plot and Save
fig.canvas.draw()
bbox = fig.get_tightbbox(fig.canvas.get_renderer())

os.makedirs("FINAL_TX_FIGS", exist_ok=True)
fig.savefig(
    "FINAL_TX_FIGS/GulfScatterTimeSeries.png",
    dpi=600,
    bbox_inches=bbox,
    facecolor="white"
)

plt.show()